# Population figure generation

Main figures and tables, followed by supplementary figures and analysis diagnostics.
Exports: `results/figures/` (PNG/PDF figures; CSV/HTML tables).
This notebook reads completed population simulations and observer fits. It does not run NEURON or refit models.

**Conventions:** grey lines = individual rats; black symbols = rat mean ? SEM.
Simulation repetitions and Monte Carlo splits are not animals.


In [ ]:
# Shared setup, validated inputs, and export helpers
from pathlib import Path
from hashlib import sha256
import shutil
import json
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from IPython.display import display, Markdown, Image

plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.20,
})


def find_repo_root():
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "data/detection/index_regime_amplitudes.csv").is_file():
            return candidate
    raise FileNotFoundError("Could not locate the population pipeline outputs")


ROOT = find_repo_root()
os.chdir(ROOT)
DETECTION = ROOT / "data/detection"
FEATURES = ROOT / "data/features"
DET_FITS = ROOT / "data/detection/behavioral_matching"
DISCRIM = ROOT / "data/discrimination/scope"
TIME_PATTERNS = ROOT / "data/time-patterns/generated"
OUT_DIR = ROOT / "results/figures"
OUT_DIR.mkdir(parents=True, exist_ok=True)

validation = json.loads((DISCRIM / "validation.json").read_text(encoding="utf-8"))
if validation.get("status") != "ready" or validation.get("failed_observer_fits", 0) != 0:
    raise RuntimeError(f"Population observer outputs are not validated: {validation}")

FREQS_DETECTION = [10.0, 20.0, 50.0, 100.0, 200.0]
FREQS_DISCRIM = [20.0, 50.0, 100.0, 200.0]
CVS = np.round(np.arange(0, 1.01, 0.1), 1)
REGIMES = ["I_first", "I_volley5", "I_volley10", "I_5", "I_10", "I_15", "I_25", "I_50", "I_all"]
FREQ_COLORS = dict(zip(FREQS_DETECTION, plt.cm.viridis(np.linspace(0.02, 0.98, len(FREQS_DETECTION)))))
FREQ_MARKERS = dict(zip(FREQS_DETECTION, ["o", "s", "D", "^", "v"]))
REGIME_COLORS = dict(zip(REGIMES, plt.cm.viridis(np.linspace(0.02, 0.98, len(REGIMES)))))
REGIME_DISPLAY = {regime: ("I_max" if regime == "I_all" else regime) for regime in REGIMES}


def display_regime(label):
    """Use manuscript-facing I_max while retaining internal I_all data keys."""
    return str(label).replace("I_all", "I_max")
WINNER_COLOR = "#6a3d9a"
COMPETITOR_COLOR = "#7f7f7f"
UPPER_COLOR = "#d89000"
EXPORT_RECORDS = []


def save_figure(fig, stem):
    for suffix in ("png", "pdf"):
        path = OUT_DIR / f"{stem}.{suffix}"
        fig.savefig(path, bbox_inches="tight")
        EXPORT_RECORDS.append({"kind": "figure", "stem": stem, "path": str(path.relative_to(ROOT))})


STATIC_DIR = ROOT / "notebooks/cache/static-figures"
STATIC_FIGURES = {
    "Figure1_PipelineOverview": {
        "png": "aa781b69aeb1e8d8ab40d3c82cfb01c9ab73afcc479d2a4d3d3aedeb9d71de0e",
        "pdf": "f9e862c7abbb35bf3534e24cc18776ce028be65f697d29c0326c056978279bd3",
    },
    "Figure2_PeriodicityDiscriminationWorkflow": {
        "png": "079449597b28fda6a798b51dce6a292bb9d6c55a91c10e4af0b8eba501a0eb1f",
        "pdf": "f51cadb8b408770f73d61b11a90b050e81780416b518ccecdf4cc63b231377d6",
    },
    "SuppFigure1_SpinalCordAnatomy": {
        "png": "cfcd44586462ae54d6ef78db4432f2b97b78b00d4d283e41ca5d813e4af1ac29",
        "pdf": "11e7ae2b8f02101829d248f16203a4ba64225f03a95249394ab3ccba6f8f7444",
    },
}


def display_static_figure(stem):
    """Publish and display an accepted design composite without regenerating it."""
    for suffix, expected_hash in STATIC_FIGURES[stem].items():
        source = STATIC_DIR / f"{stem}.{suffix}"
        if not source.is_file() or sha256(source.read_bytes()).hexdigest() != expected_hash:
            raise ValueError(f"Missing or modified accepted figure: {source}")
        target = OUT_DIR / source.name
        shutil.copyfile(source, target)
        EXPORT_RECORDS.append({"kind": "figure", "stem": stem,
                               "path": str(target.relative_to(ROOT))})
    display(Image(filename=str(OUT_DIR / f"{stem}.png")))


def save_table(frame, stem):
    path = OUT_DIR / f"{stem}.csv"
    frame.to_csv(path, index=False)
    EXPORT_RECORDS.append({"kind": "table", "stem": stem, "path": str(path.relative_to(ROOT))})
    return path


def sem(values):
    values = pd.to_numeric(pd.Series(values), errors="coerce").dropna().to_numpy(float)
    return float(values.std(ddof=1) / np.sqrt(len(values))) if len(values) > 1 else 0.0


index_regimes = pd.read_csv(DETECTION / "index_regime_amplitudes.csv", low_memory=False)
equivalence = pd.read_csv(FEATURES / "population_regime_response_equivalence.csv", low_memory=False)
population_features = pd.read_csv(FEATURES / "population_cv0_based_features.csv", low_memory=False)
population_evidence = pd.read_csv(FEATURES / "population_cv0_template_evidence.csv", low_memory=False)
recruitment = pd.read_csv(DETECTION / "population_recruitment_outcomes.csv", low_memory=False)
phase_summary = pd.read_csv(DETECTION / "reference_phase_threshold_summary.csv", low_memory=False)
detection_curves = pd.read_csv(DET_FITS / "detection_curves.csv", low_memory=False)
detection_metrics = pd.read_csv(DET_FITS / "detection_metrics.csv", low_memory=False)
detection_weights = pd.read_csv(DET_FITS / "detection_weights.csv", low_memory=False)
observer_curves = pd.read_csv(DISCRIM / "population_discrimination_curves.csv", low_memory=False)
observer_curves_mc = pd.read_csv(DISCRIM / "merged_curves.csv", low_memory=False)
observer_results_mc = pd.read_csv(DISCRIM / "merged_results.csv", low_memory=False)
observer_rmse = pd.read_csv(DISCRIM / "population_observer_rmse.csv", low_memory=False)
observer_weights = pd.read_csv(DISCRIM / "population_observer_weights.csv", low_memory=False)
rate_share = pd.read_csv(DISCRIM / "population_effective_rate_share.csv", low_memory=False)

print(f"Repository: {ROOT}")
print(f"Exports: {OUT_DIR}")
print(f"Validated v5 observer fits: {validation['observer_fits']:,}; MC splits: {validation['expected_mc']}")

import sys
sys.path.insert(0, str(ROOT / 'src'))
from figure_generation.figure_model import PopulationFigures
figures = PopulationFigures(ROOT, OUT_DIR)


## Figure 1 — computational modeling pipeline

The accepted Adobe/BioRender composite is displayed below from `notebooks/cache/static-figures/`.
Its PNG is byte-identical to the image embedded in the accepted manuscript. The PDF is the
matching artwork export. The following cell also regenerates population sampling and response
panels as auditable component assets; it does not rebuild the manually assembled composite.


In [ ]:
from figure_generation.fig1_sampling import sampling_component
partial_figures = [sampling_component(figures), *figures.partials(freq=100, rep=0)]
for partial in partial_figures:
    display(partial)
    plt.close(partial)

In [ ]:
display_static_figure("Figure1_PipelineOverview")


## Figure 2 — periodicity discrimination workflow

The accepted BioRender/Adobe composite is displayed below from `notebooks/cache/static-figures/`.
Its PNG is byte-identical to the image embedded in the accepted manuscript. The PDF is the
matching artwork export. The following cell also regenerates the model example and raster
components from the saved population outputs; it does not rebuild the manually assembled composite.


In [ ]:
fig = figures.fig2(freq=100)
display(fig)
plt.close(fig)
from figure_generation.fig2_rasters import raster_components
for partial in raster_components(figures):
    display(partial)
    plt.close(partial)

In [ ]:
display_static_figure("Figure2_PeriodicityDiscriminationWorkflow")


## Figure 3 — detection capacity and the fitted per-CV mixtures

a: Median absolute index thresholds. b: Individual rats, mean ± SEM, one global fixed regime
(green), per-frequency mixtures held constant across CV (purple dashed), and the
per-CV capacity fit (orange squares) to the six measured points. All three use
the same mean behavioral targets and frequency-specific biases.
c: Six discrete mixture bars per frequency, not interpolation of unobserved weights.
Behavior panels use frequency-specific y ranges, retaining all rats and SEM endpoints;
the caption must note the differing ranges. Gray borders separate measured-CV bars; no grid is drawn.

Behavior is normalized within rat and frequency to that rat's maximum threshold across CV.
The detection objective uses mean amplitude curves; row a is explicitly a median description.
The highly flexible per-CV fit is a capacity bound, not a held-out prediction or a uniquely
identified biological mixture. Normalized threshold curves remain available in the original notebook.


In [ ]:
fig = figures.fig3()
plt.show()

## Figure 4 — independently fitted evidence families

a: Independent Rate, Timing, and Hybrid predictions under per-frequency mixtures and block-specific
decision parameters, with behavioral mean/SEM and individual rats. Each family optimizes its own mixtures,
gains and biases. Hybrid is bold as the focal model, not a claim of superiority to every model.
b: Per-frequency hybrid CV-averaged rate reliance; dots are means and tails central 95% ranges across
simulation splits, not animal confidence intervals or repetition-level bootstrap intervals.
c: Each family's own frequency-specific mixture, shared across Std0/Std1 within that family.
These are NOT the former hybrid-mixture-locked controls.

Version B (grouped bars) is the selected panel-c display. Each retained regime has Rate (red),
Hybrid (purple), and Timing (blue) bars; joined x-axis labels identify frequency-specific
response-equivalent regimes. All frequency columns share the same mixture-weight y range.
Alternative stacked and categorical-line displays remain in `figures_v3/exploratory_fig4c/`
for reference and are not additional manuscript outputs.

This is the preferred constrained structure versus the crossed alternative; the per-panel hybrid
remains a less-constrained benchmark in S9/Table 1. The primary is not the lowest-RMSE structure
overall. Independent Timing closely matches Hybrid in Std1; their small error difference is not
robust to the corrected-resampling sensitivity. Archived statistical exports retain the conditional
locked-mixture diagnostics separately, without displaying them as main-figure controls.


In [ ]:
# Selected version B: independent cue models with grouped mixture bars.
fig = figures.fig4_reliance_preview(mixture_style='grouped', locked_mixtures=False)
display(fig)
plt.close(fig)


## Table 1 — model performance and rate reliance

One table combines the five models, nominal parameter counts, overall/block RMSE,
heuristic ΔAIC*/ΔBIC*, and block-mean rate reliance. Model names refer to mixture sharing:
per-frequency mixtures use block-specific gains/intercepts; per-block mixtures use
frequency-specific gains/intercepts; per-panel mixtures use panel-specific parameters.
Overall RMSE pools all frequency–block–CV points; block RMSE averages four frequency RMSEs.
Rate/timing-only reliance is fixed at 1/0 by construction. Behavioral animals are not held out.
Full numerical diagnostic exports remain available; they are not numbered supplementary tables.


In [ ]:
from figure_generation.tables import consolidated_table
display(consolidated_table(figures))

## Statistical comparisons — population observer

These tables extend the finalized single-axon release's statistical workflow without changing
the population fits, figures, or Table 1. **All inference is conditional on simulations and saved
observers, not independent animals.** The same behavioral targets were used during fitting.

RMSE: two-sided paired Wilcoxon tests on each split's mean of four frequency-specific RMSEs,
Holm-adjusted over all three model pairs within each family/block. Repeated-split p-values describe
partition stability. The additional corrected-resampled t p-values/95% intervals are approximate
dependence-aware sensitivities, **not** validated confirmatory inference for this observer.
Retained mixture-locked diagnostics are kept separate from independently optimized cue models (Fig. 4).
Missing source files are reported explicitly; summary means/SEMs cannot reconstruct paired tests.

Rate reliance: each held-out repetition is evaluated with its saved observer; its CV-integrated
reliance is averaged over test appearances before testing (100 reps, not 200 splits or 420 axons).
This is a different estimand from Fig. 4's reliance of split-averaged evidence, shown below for audit.
Block contrasts use paired sign flips; frequency contrasts use unpaired label permutations.
Both use two-sided absolute-mean statistics, 10,000 permutations and 5,000 bootstrap draws.
Bonferroni families contain 8 block and 24 frequency contrasts across primary/per-panel hybrids;
Holm is also exported as a sensitivity. Bootstrap intervals are pointwise, not simultaneous.

These repetition tests are **exploratory and conditional on the saved fits**: shared-training
dependence/fit uncertainty is not removed by averaging test appearances. Sign flips require
symmetric/exchangeable null differences; frequency permutations require exchangeability across
frequency samples. Nonsignificance does not establish equivalence. No outer refit bootstrap is run.
Full methods, software versions, hashes, splits audit, and numerical observations are exported
under `figures_v3/statistics/`. Existing Fig. 4 reliance values remain unchanged.

References: [Wilcoxon assumptions](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.wilcoxon.html),
[permutation assumptions](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.permutation_test.html),
[corrected resampling sensitivity](https://scikit-learn.org/stable/auto_examples/model_selection/plot_grid_search_stats.html).


In [ ]:
# Standalone/re-runnable; no figure regeneration or model fitting.
from pathlib import Path
import sys
from IPython.display import display, Markdown
_stats_root = next(p for p in [Path.cwd(), *Path.cwd().parents]
                   if (p / 'src').is_dir())
if str(_stats_root / 'src') not in sys.path:
    sys.path.insert(0, str(_stats_root / 'src'))
from figure_generation.statistics import run_statistics
statistics_output = _stats_root / 'results/figures/statistics'
statistics_tables = run_statistics(_stats_root, statistics_output)
display(statistics_tables['availability'])
display(Markdown('Reconstruction checks passed for every included train/test fit. '
                 'Missing independent controls, if listed above, have not been replaced by locked controls.'))
if 'figures' in globals():
    for name in statistics_tables:
        record = dict(kind='table', stem=f'statistics/{name}')
        if record not in figures.records:
            figures.records.append(record)


In [ ]:
display(Markdown('### Prediction error: paired split comparisons'))
_rmse = statistics_tables['rmse_tests']
for _family, _table in _rmse.groupby('family', sort=False):
    display(Markdown(f'**{_family.replace("_", " ")}** — negative difference favors model A. '
                     'Wilcoxon: conditional split stability; corrected t: approximate sensitivity.'))
    display(_table[['block', 'model_a', 'model_b', 'n_paired_mc', 'mean_a', 'mean_b',
                    'difference_a_minus_b', 'wilcoxon_p_holm', 'corrected_t_p_holm',
                    'corrected_ci_low', 'corrected_ci_high']].style.format(precision=5).format(
                        {'wilcoxon_p_holm': '{:.3g}', 'corrected_t_p_holm': '{:.3g}'}))


In [ ]:
display(Markdown('### Rate reliance: repetition-level comparisons'))
_reliance = statistics_tables['reliance_tests']
for _family, _table in _reliance.groupby('family', sort=False):
    display(Markdown(f'**{_family.replace("_", " ")}** — positive difference favors the first block/frequency. '
                     'Conditional exploratory tests; pointwise bootstrap intervals.'))
    display(_table[['model', 'freq', 'block', 'comparison', 'n_a', 'n_b', 'mean_a', 'mean_b',
                    'difference_a_minus_b', 'ci_low', 'ci_high', 'p_bonferroni']].style.format(
                        precision=4, na_rep='—').format({'p_bonferroni': '{:.3g}'}))
display(Markdown('**Estimand check:** per-repetition AUC versus the existing split-mean-evidence AUC. '
                 'Differences here do not change Fig. 4/Table 1.'))
display(statistics_tables['reliance_estimand_comparison'].round(4))


### Interpretation of statistical comparisons

- Prediction error and model complexity answer different questions.
- Small differences can be stable across simulation splits without establishing biological importance.
- Block reliance tests compare Std0 with Std1; they do not test either against 0.5.
- Very small permutation p-values are limited by the finite resampling resolution.
- The statistical tests are conditional on the saved model fits and simulated repetitions.


## Supplementary Figure 1 — computational model domains

The accepted COMSOL/Adobe composite is displayed below from `notebooks/cache/static-figures/`.
Its PNG is byte-identical to the image embedded in the accepted manuscript; the PDF is the
matching artwork export.

## Supplementary Figure 2 — index regime amplitudes

Amplitude curves show medians plus all individual stimulus-train repetitions, not confidence intervals.
Small deterministic horizontal offsets/jitter improve visibility without altering amplitudes.
The internal key `I_all` displays as $I_{max}$, selected from the initial peak/plateau branch.


In [ ]:
display_static_figure("SuppFigure1_SpinalCordAnatomy")


In [ ]:
from figure_generation.supplementary import index_counts
fig = index_counts(figures, amplitudes=True)
display(fig)
plt.close(fig)

## Supplementary Figure 3 - true-CV index-axon spike counts

Median lines and faint individual-repetition points replace the overlapping count clouds.
Points have small display-only horizontal frequency offsets and deterministic jitter; values
are not clipped. The top three regimes show counts; percentage-defined regimes show percent
of nominal input pulses. Quantiles remain available in the exported summary table.
Separate central-80% previews retain only pointwise 10th–90th-percentile observations for display;
their median lines still use all repetitions. Those previews are not promoted here.


In [ ]:
from figure_generation.supplementary import index_counts
fig = index_counts(figures, 'true')
display(fig)
plt.close(fig)

## Supplementary Figure 4 - CV0-anchored index-axon spike counts

The same display convention as S3, using CV0-anchored responses rather than true-CV amplitudes.


In [ ]:
from figure_generation.supplementary import index_counts
fig = index_counts(figures, 'cv0')
display(fig)
plt.close(fig)

## Supplementary Figure 5 - spatial recruitment and population output

The accepted compact conditional-mean/opacity composite is the sole S5 output. Columns show the
observer frequencies (20, 50, 100, 200 Hz). Exploratory and locked versions remain outside
the notebook; no simulations or observer models are refitted.

**Caption notes.** a: Spatial responses at each repetition's index-defined $I_{max}$ amplitude,
aggregated across all 100 repetitions. The displayed CV maximizes mean true-CV recruited-axon
count at each frequency (currently CV = 1 for all four); both amplitude bases use that same CV.
This is not maximal occupied-location coverage or an extreme individual repetition. True-CV amplitudes are defined
at the displayed CV; CV0-based amplitudes are anchored at CV0. Ring color represents the
mean spike count **conditional on at least one spike**, mapped to the nearest count-equivalent
operating-regime anchor. These are approximate output categories, not independently measured
per-axon thresholds or plateaus. The $I_{max}$ color anchor is the mean index-axon count at
the amplitude-defining CV; equal-count aliases retain the first canonical label. Fill opacity
equals the fraction of repetitions with spikes; never-active halves are unfilled. This is an
ensemble summary, not one simultaneously observed population.

Six schematic rings, inner to outer, represent diameters 5.7, 8.7, 10, 11.5, 12.8, and 14 micrometers;
left/right halves represent nodal phases 1/0. The black outline identifies the index axon.
S/M/D denote superficial/middle/deep sampling rows; columns and ring sizes are schematic,
not anatomical distances or diameter-scaled glyphs.

b/c: All individual recruited-axon counts and total population spike counts with median lines
across 100 repetitions, for all CVs and index regimes. True-CV and CV0-based amplitudes occupy
separate rows. Individual points have display-only horizontal offsets; medians remain at actual CVs.
Logarithmic y ranges differ by frequency and amplitude basis, retain every positive observation,
and extend to an upper anchoring tick. Only powers of ten are labeled; intermediate ticks remain visible.
Exact zeros appear on separate linear strips, with no pseudocounts. Straight connecting guides
join adjacent zero/positive medians across the axes break; they do not represent logarithmic
interpolation through zero. Individual repetitions remain unconnected.
The horizontal regime palette is shared across panels; no additional keys are drawn.

Export: `suppfig5_population_recruitment.png/.pdf`; audited source tables and detailed
color definitions: `suppfig5_population_recruitment_data/_data/`.


True-CV means amplitudes determined separately at each CV; CV0-based means
amplitudes determined at CV = 0 and held fixed across CV. Both use
index-axon regime amplitudes applied to the entire population.


In [ ]:
from figure_generation.spatial_variability import manuscript_figure
fig = manuscript_figure(figures)
display(fig)
plt.close(fig)

## Supplementary Figure 6 - population feature engineering example

200 Hz, I_50, repetition 0. Single-axon figure conventions are used for labels, primary/secondary
feature axes, and dotted/dashed spike/burst lines. Viridis CV traces and burst shading are retained.
The raster line width reflects population spike count per bin; the envelope is binary support,
not recruitment magnitude. Symmetric padding around 0–2000 ms retains the response tail.
Secondary axes have visible right spines and independent headroom, following the old single-axon
figure. Bottom rate evidence is the raw bounded
feature transform, as before; it is not a newly rescaled version of the model evidence.


In [ ]:
from figure_generation.supplementary import feature_example
fig = feature_example(figures)
display(fig)
plt.close(fig)

## Supplementary Figure 7 - population evidence curves

The promoted four-by-four mean/SEM curves are the manuscript output:
frequency columns and rate/timing-by-window rows. Values are means across simulated repetitions,
not across animals or model-fitting splits.
Response-equivalent regimes use their retained representatives. Red/blue paired regime swatches
and dotted/dashed window coding are preserved. The heatmap remains available via
`figures.evidence_heatmaps()` but is not displayed here. Clouds show mean plus/minus SEM across
simulated repetitions. Separate exploratory outputs
compare median/IQR, mean/SEM, and median/individual repetitions. IQR describes repetition spread;
SEM describes uncertainty of the mean, not that spread. None changes the evidence transformation.
Set `style='median_iqr'`, `'mean_sem'`, or `'individual_reps'` in the helper to inspect alternatives.


In [ ]:
# Optional diagnostic heatmap: figures.evidence_heatmaps(); not a manuscript output.

In [ ]:
from figure_generation.evidence_variability import evidence_curves
fig = evidence_curves(figures, style='mean_sem', stem='suppfig7_population_evidence_curves')
display(fig)
plt.close(fig)

## Supplementary Figure 8 - sub-refractory intervals

The x-axis is the fraction of inter-pulse intervals shorter than 1.10 ms.
Only observed points are shown; the $y=1-x$ reference line has been removed. Describe the overall
relationship without treating a one-lost-spike-per-short-IPI approximation as a mechanistic prediction.
Scatter position alone does not identify a mechanism or demonstrate refractory violations.


Caption: **Supplementary Figure 8: Sub-Refractory Inter-Pulse Intervals.**
Fraction of expected spikes in the index axon at $I_{max}$ versus the fraction of
inter-pulse intervals < 1.10 ms. Expected spikes equal the nominal input pulse
count. Each point represents one simulated repetition; colors and symbols indicate
stimulation frequency.


In [ ]:
# Retained mechanistic validation: sub-refractory intervals versus I_max spike reproduction
ABS_REF_MS = 1.10
# Versioned because the finalized single-axon implementation used a strict < cutoff.
ISI_CACHE = ROOT / "notebooks/cache/cache_index_sub_refractory_intervals_lt1p10ms.csv"


def compute_isi_summary():
    records = []
    abs_ref_samples = int(round(ABS_REF_MS * 20000.0 / 1000.0))
    for cv in CVS:
        for freq in FREQS_DETECTION:
            path = TIME_PATTERNS / f"cv{cv:.1f}_f{int(freq)}_waveforms.npz"
            with np.load(path) as bundle:
                for rep_label in bundle.files:
                    waveform = np.asarray(bundle[rep_label])
                    active = waveform < 0
                    starts = np.flatnonzero(active[1:] & ~active[:-1]) + 1
                    if active.size and active[0]:
                        starts = np.r_[0, starts]
                    intervals = np.diff(starts)
                    records.append({
                        "cv": float(cv), "freq": float(freq), "rep_label": rep_label,
                        "n_intervals": len(intervals),
                        "fraction_ipi_lt_1p10ms": float(np.mean(intervals < abs_ref_samples)) if len(intervals) else np.nan,
                    })
    return pd.DataFrame(records)


if ISI_CACHE.is_file():
    isi = pd.read_csv(ISI_CACHE)
else:
    isi = compute_isi_summary()
    ISI_CACHE.parent.mkdir(parents=True, exist_ok=True)
    isi.to_csv(ISI_CACHE, index=False)

i_max = index_regimes[index_regimes["thresh_type"].eq("I_all")].copy()
i_max["fraction_expected_spikes"] = i_max["N_max_spikes"] / i_max["n_input_pulses"]
sub_ref = i_max.merge(isi, on=["cv", "freq", "rep_label"], how="left", validate="one_to_one")

sub_ref["short_intervals_per_input_pulse"] = sub_ref["fraction_ipi_lt_1p10ms"] * sub_ref["n_intervals"] / sub_ref["n_input_pulses"]
fig, ax = plt.subplots(figsize=(8.3, 6.2))
for freq in reversed(FREQS_DETECTION):
    panel = sub_ref[np.isclose(sub_ref["freq"], freq)]
    ax.scatter(panel["fraction_ipi_lt_1p10ms"], panel["fraction_expected_spikes"],
               s=28, alpha=0.55, marker=FREQ_MARKERS[freq], color=FREQ_COLORS[freq], label=f"{int(freq)} Hz")
x_max = float(sub_ref["fraction_ipi_lt_1p10ms"].max())
x_reference = np.linspace(0, max(0.26, x_max), 200)

ax.set_xlabel(f"Fraction of intervals < {ABS_REF_MS:.2f} ms")
ax.set_ylabel("Fraction of expected spikes")
ax.set_xlim(-0.005, max(0.26, x_max + 0.01))
ax.set_ylim(0.7, 1.03)
ax.legend(frameon=False, title="Frequency", loc="lower left")

fig.tight_layout()
save_figure(fig, "suppfig8_sub_refractory_intervals")
plt.show()
save_table(sub_ref[["freq", "cv", "rep_index", "rep_label", "fraction_ipi_lt_1p10ms", "fraction_expected_spikes", "short_intervals_per_input_pulse"]],
           "supp_index_sub_refractory_intervals_data")


## Supplementary Figure 9 — hybrid mixture/readout structures

Panel a compares only three hybrid observers: per-frequency mixtures/block-specific gains and intercepts,
per-block mixtures/frequency-specific gains and intercepts, and per-panel mixtures/per-panel readouts.
The first two are constrained sharing hypotheses; the per-panel model is a less-constrained
benchmark, not a post hoc exclusion from the model comparison. Primary is repeated as a reference.
Panel b shows their CV-averaged rate reliance and central 95% ranges across simulation splits.
These are not the repetition-level observations/intervals used in the permutation analysis.

Independent Rate/Timing curves now appear in main Fig. 4 and are omitted here. Locked-mixture
controls are retained as diagnostics in the results/statistics, not plotted in this figure.
Legend RMSE values are pooled overall simulation-test RMSEs averaged across splits, as in Table 1;
they are not block-mean RMSE or RMSE of the displayed mean curve. Table 1 retains all five models,
including the per-panel model's lower error/AIC-based score and primary's lower BIC-based score.
The primary and per-panel hybrids agree on the direction of the Std0/Std1 difference, not on a
common frequency trend or identical reliance magnitudes. No statistical stars are added here.


In [ ]:
from figure_generation.supplementary import model_comparison
fig = model_comparison(figures)
display(fig)
plt.close(fig)

## Analysis diagnostics

The following tables retain numerical diagnostics for detection scope, rate reliance, and nodal phase.
They are included to make the figure inputs and statistical comparisons auditable.


In [ ]:
# Matched-location nodal-phase sensitivity table
# The former I_max-only recruitment panel is redundant with the all-regime
# recruitment figure below, so retain the anatomical sensitivity result as a table.
diameter_phase = phase_summary[phase_summary["stratum_type"].eq("diameter_um")].copy()
diameter_phase["stratum"] = pd.to_numeric(diameter_phase["stratum"])
diameter_phase = diameter_phase.sort_values("stratum")

nodal_separation_table = pd.DataFrame({
    "Axon diameter (um)": diameter_phase["stratum"],
    "Matched pairs": diameter_phase["matched_pairs"].astype(int),
    "Phase-1 lower (n)": diameter_phase["ph1_lower_count"].astype(int),
    "Median gap (uA)": diameter_phase["median_gap_uA"],
    "Median gap (%)": diameter_phase["median_gap_percent"],
    "IQR lower (%)": diameter_phase["q25_gap_percent"],
    "IQR upper (%)": diameter_phase["q75_gap_percent"],
    "Minimum gap (%)": diameter_phase["min_gap_percent"],
    "Maximum gap (%)": diameter_phase["max_gap_percent"],
}).reset_index(drop=True)

display(Markdown("### Nodal-phase threshold separation by axon diameter"))
display(nodal_separation_table.style.format({
    "Axon diameter (um)": "{:.1f}", "Median gap (uA)": "{:.2f}",
    "Median gap (%)": "{:.2f}", "IQR lower (%)": "{:.2f}",
    "IQR upper (%)": "{:.2f}", "Minimum gap (%)": "{:.2f}",
    "Maximum gap (%)": "{:.2f}",
}))
save_table(nodal_separation_table, "supp_nodal_phase_threshold_separation")


## Export manifest

The manifest lists generated figure and table exports together with the accepted static
composites. Static composites are copied from the tracked notebook cache, not recreated by Python.


In [ ]:
manifest = pd.concat([pd.DataFrame(EXPORT_RECORDS), pd.DataFrame(figures.records)], ignore_index=True)
manifest.to_csv(OUT_DIR / 'figure_manifest_v3.csv', index=False)
display(manifest[['kind', 'stem']])